In [2]:
%pip install scikit-learn imbalanced-learn pandas numpy

  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl (8.0 MB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install tensorflow

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.2-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached grpcio-1.80.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Silencia advertencias internas de TensorFlow

import warnings
warnings.filterwarnings('ignore', category=UserWarning)  # Silencia los avisos del scaler

import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

In [4]:
# Carga del dataset original
df = pd.read_csv('../data/titanic.csv')

# Eliminación de columnas no representativas para la RNA
columnas_a_eliminar = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df = df.drop(columns=columnas_a_eliminar)

# Tratamiento de valores nulos 
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Codificación de variables categóricas
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Convertir booleanos resultantes a enteros (0 y 1) para la RNA
df = df.astype({col: 'int32' for col in df.select_dtypes(include='bool').columns})

# Separar características (X) y etiqueta objetivo (y)
X = df.drop(columns=['Survived'])
y = df['Survived']

# Escalado de características numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Balanceo de datos mediante Over-sampling (SMOTE)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print(f"Distribución original de clases: {np.bincount(y)}")
print(f"Distribución tras SMOTE: {np.bincount(y_resampled)}")

Distribución original de clases: [549 342]
Distribución tras SMOTE: [549 549]


In [5]:
print("--- División del conjunto de datos ---")

# Dividir en Entrenamiento (70%) y una porción Temporal (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_resampled, y_resampled, test_size=0.30, random_state=42
)

# Dividir la porción temporal en Validación (15%) y Prueba (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Registros de Entrenamiento: {X_train.shape[0]}")
print(f"Registros de Validación: {X_val.shape[0]}")
print(f"Registros de Prueba: {X_test.shape[0]}")

--- División del conjunto de datos ---
Registros de Entrenamiento: 768
Registros de Validación: 165
Registros de Prueba: 165


In [6]:
print("--- Entrenando Modelo 1: Arquitectura Simple ---")

# Definición de la arquitectura
model_1 = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  # Capa oculta simple
    Dense(1, activation='sigmoid')                                  # Capa de salida binaria
])

# Configuración del optimizador
learning_rate_1 = 0.01
optimizador_1 = Adam(learning_rate=learning_rate_1)

model_1.compile(
    optimizer=optimizador_1,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_model_1 = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('mejor_modelo_1.keras', save_best_only=True, monitor='val_loss')
]

# Ejecución del entrenamiento
inicio_tiempo = time.time()
historia_1 = model_1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks_model_1,
    verbose=1
)
print(f"Tiempo de entrenamiento de Modelo 1: {time.time() - inicio_tiempo:.2f} segundos")

# --- EVALUACIÓN MODELO 1 ---
print("\n--- Evaluando Modelo 1 en datos de Prueba (Test) ---")
predicciones_prob = model_1.predict(X_test)
predicciones = (predicciones_prob > 0.5).astype(int)

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, predicciones))
print("\nReporte de Clasificación:")
print(classification_report(y_test, predicciones))

--- Entrenando Modelo 1: Arquitectura Simple ---
Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7305 - loss: 0.5706 - val_accuracy: 0.7939 - val_loss: 0.4854
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8073 - loss: 0.4605 - val_accuracy: 0.8182 - val_loss: 0.4376
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8034 - loss: 0.4439 - val_accuracy: 0.8242 - val_loss: 0.4234
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8203 - loss: 0.4338 - val_accuracy: 0.8061 - val_loss: 0.4207
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8190 - loss: 0.4272 - val_accuracy: 0.8121 - val_loss: 0.4218
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8177 - loss: 0.4257 - val_accuracy: 0.8061 - val_loss: 0.4200
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8112 - loss: 0.4208 - val_accuracy: 0.8061 - val_loss: 0.4196
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy

In [7]:
print("--- Entrenando Modelo 2: Arquitectura Intermedia ---")

# Definir la arquitectura
model_2 = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)), # Capa 1
    Dense(16, activation='relu'),                                  # Capa 2
    Dense(1, activation='sigmoid')                                 # Salida
])

# Configuración
learning_rate_2 = 0.001
model_2.compile(
    optimizer=Adam(learning_rate=learning_rate_2),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_model_2 = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    ModelCheckpoint('mejor_modelo_2.keras', save_best_only=True, monitor='val_loss')
]

# Entrenamiento
inicio_tiempo = time.time()
historia_2 = model_2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=32,
    callbacks=callbacks_model_2,
    verbose=1
)
print(f"Tiempo de entrenamiento de Modelo 2: {time.time() - inicio_tiempo:.2f} segundos")

# --- EVALUACIÓN MODELO 2 ---
print("\n[MÉTRICAS MODELO 2]")
predicciones_prob_2 = model_2.predict(X_test)
predicciones_2 = (predicciones_prob_2 > 0.5).astype(int)

print("Matriz de Confusión:")
print(confusion_matrix(y_test, predicciones_2))
print("\nReporte de Clasificación:")
print(classification_report(y_test, predicciones_2))

--- Entrenando Modelo 2: Arquitectura Intermedia ---
Epoch 1/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5846 - loss: 0.6696 - val_accuracy: 0.6970 - val_loss: 0.6318
Epoch 2/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7148 - loss: 0.6056 - val_accuracy: 0.7152 - val_loss: 0.5812
Epoch 3/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7396 - loss: 0.5576 - val_accuracy: 0.7697 - val_loss: 0.5378
Epoch 4/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7799 - loss: 0.5169 - val_accuracy: 0.7939 - val_loss: 0.5057
Epoch 5/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7865 - loss: 0.4861 - val_accuracy: 0.8000 - val_loss: 0.4764
Epoch 6/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7943 - loss: 0.4619 - val_accuracy: 0.7939 - val_loss: 0.4576
Epoch 7/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8008 - loss: 0.4462 - val_accuracy: 0.8000 - val_loss: 0.4473
Epoch 8/120
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accu

In [8]:
print("--- Entrenando Modelo 3: Complejo con Regularización ---")

# Definir la arquitectura
model_3 = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Configuración
learning_rate_3 = 0.001
model_3.compile(
    optimizer=Adam(learning_rate=learning_rate_3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_model_3 = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ModelCheckpoint('mejor_modelo_3.keras', save_best_only=True, monitor='val_loss')
]

# Entrenamiento
inicio_tiempo = time.time()
historia_3 = model_3.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks_model_3,
    verbose=1
)
print(f"Tiempo de entrenamiento de Modelo 3: {time.time() - inicio_tiempo:.2f} segundos")

# --- EVALUACIÓN MODELO 3 ---
print("\n[MÉTRICAS MODELO 3]")
predicciones_prob_3 = model_3.predict(X_test)
predicciones_3 = (predicciones_prob_3 > 0.5).astype(int)

print("Matriz de Confusión:")
print(confusion_matrix(y_test, predicciones_3))
print("\nReporte de Clasificación:")
print(classification_report(y_test, predicciones_3))

--- Entrenando Modelo 3: Complejo con Regularización ---
Epoch 1/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6576 - loss: 0.6407 - val_accuracy: 0.7576 - val_loss: 0.5814
Epoch 2/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7513 - loss: 0.5645 - val_accuracy: 0.7758 - val_loss: 0.4993
Epoch 3/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7656 - loss: 0.5051 - val_accuracy: 0.7758 - val_loss: 0.4637
Epoch 4/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7747 - loss: 0.4912 - val_accuracy: 0.7758 - val_loss: 0.4469
Epoch 5/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7865 - loss: 0.4714 - val_accuracy: 0.7818 - val_loss: 0.4362
Epoch 6/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7943 - loss: 0.4833 - val_accuracy: 0.8000 - val_loss: 0.4344
Epoch 7/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7943 - loss: 0.4675 - val_accuracy: 0.8182 - val_loss: 0.4282
Epoch 8/150
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step -

In [9]:
print("--- Simulando el Modelo en Producción (Inferencia) ---")

# Columnas que el scaler conoce
columnas = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']

# Datos de pasajeros a evaluar (¡Cámbialos cuando quieras!)
nuevos_pasajeros = [
    [1, 29.0, 0, 0, 80.0, 0, 0, 0],  # Pasajera A (Rose)
    [3, 22.0, 0, 0, 7.5, 1, 0, 1]   # Pasajero B (Jack)
]

# 1. Convertimos a DataFrame para mantener consistencia
df_nuevos = pd.DataFrame(nuevos_pasajeros, columns=columnas)

# 2. Escalamos usando el scaler vivo de la Celda 2
datos_escalados = scaler.transform(df_nuevos)

# 3. Hacemos la predicción usando el 'model_2' entrenado en la Celda 5
probabilidades = model_2.predict(datos_escalados, verbose=0).flatten()

# 4. Mostrar resultados estéticos
nombres = ["Pasajera Ficticia A (Perfil Alta Probabilidad)", "Pasajero Ficticio B (Perfil Baja Probabilidad)"]

for i, nombre in enumerate(nombres):
    prob = probabilidades[i]
    print(f"\nAnálisis para: {nombre}")
    print(f"-> Probabilidad matemática de salvarse: {prob * 100:.2f}%")
    if prob > 0.5:
        print("-> Veredicto de la IA: ¡SOBREVIVE! 🚢✨")
    else:
        print("-> Veredicto de la IA: NO SOBREVIVE. 🌊")

--- Simulando el Modelo en Producción (Inferencia) ---

Análisis para: Pasajera Ficticia A (Perfil Alta Probabilidad)
-> Probabilidad matemática de salvarse: 97.90%
-> Veredicto de la IA: ¡SOBREVIVE! 🚢✨

Análisis para: Pasajero Ficticio B (Perfil Baja Probabilidad)
-> Probabilidad matemática de salvarse: 14.72%
-> Veredicto de la IA: NO SOBREVIVE. 🌊
